In [29]:
import numpy as np
import pandas as pd
import itertools

# ----------------------------
# Load + parse (as you already do)
# ----------------------------
df = pd.read_csv("chi_table_with_errors.csv", index_col=0)

vals = df.applymap(lambda s: float(s.split(" ± ")[0]))
errs = df.applymap(lambda s: float(s.split(" ± ")[1]))

# ----------------------------
# Helpers
# ----------------------------
def chi(i, j):
    return float(vals.loc[i, j])

def chi_err(i, j):
    return float(errs.loc[i, j])

def combined_sigma(e1, e2):
    return float(np.sqrt(e1**2 + e2**2))

def is_frustrated(A, B, C, k_margin=0.0):
    """
    We say frustrated if:
        χ_BC > χ_AB and χ_AC
    where "≈" means within k_equal * combined_sigma.
    Optionally require a margin k_margin*sigma for the '<' comparison.
    """
    chi_ab, e_ab = chi(A, B), chi_err(A, B)
    chi_bc, e_bc = chi(B, C), chi_err(B, C)
    chi_ac, e_ac = chi(A, C), chi_err(A, C)

    # "<" test with optional uncertainty margin
    if chi_bc > chi_ab and chi_bc > chi_ac:
        frustrated = True
    else:
        frustrated = False
    return frustrated

def within_bounds(A, B, C):
    chi_ab, e_ab = chi(A, B), chi_err(A, B)
    chi_bc, e_bc = chi(B, C), chi_err(B, C)
    chi_ac, e_ac = chi(A, C), chi_err(A, C)

    if chi_ab > 0 and chi_bc > 0 and chi_ac > 0:
        first = chi_ab / chi_ac >= 1/32 and chi_ab / chi_ac <= 32
        second = chi_ab / chi_bc >= 1 and chi_ab / chi_bc <= 32
        third = chi_ac / chi_bc >= 1 and chi_ac / chi_bc <= 32
        if first and second and third:
            return True

# ----------------------------
# Build candidate sets
# ----------------------------
all_names = list(vals.index)

C_candidates = [n for n in all_names if n.startswith("4")]
AB_candidates = [n for n in all_names if n[0] in {"1", "2", "3"}]  # 1a..,2a..,3a..

# ----------------------------
# Enumerate triples
# ----------------------------
k_margin = 0.0   # require χ_AC to be smaller by this many sigmas (0 = just smaller)

nonfrust = []
frust = []
good = []

for C in C_candidates:
    for A, B in itertools.permutations(AB_candidates, 2):  # ordered A,B
        if A == B:
            continue

        chi_ab = chi(A, B)
        chi_bc = chi(B, C)
        chi_ac = chi(A, C)

        record = {
            "A": A, "B": B, "C": C,
            "chi_AB": chi_ab, "err_AB": chi_err(A, B),
            "chi_BC": chi_bc, "err_BC": chi_err(B, C),
            "chi_AC": chi_ac, "err_AC": chi_err(A, C),
        }

        if is_frustrated(A, B, C, k_margin=k_margin):
            frust.append(record)
        else:
            nonfrust.append(record)
        
        if within_bounds(A, B, C):
            good.append(record)

# ----------------------------
# Print results
# ----------------------------
print(f"Non-frustrated triples: {len(nonfrust)}")
print(f"Frustrated triples: {len(frust)}")
print(f"Triples within bounds: {len(good)}")

all_A = []
all_B = []
all_C = []
for r in good:
    print(
        f'A: {r["A"]}, B: {r["B"]}, C: {r["C"]} | '
        f'χ_AB={r["chi_AB"]:.6f}, '
        f'χ_BC={r["chi_BC"]:.6f}, '
        f'χ_AC={r["chi_AC"]:.6f} '
    )
    if r["A"] not in all_A:
        all_A.append(r["A"])
    if r["B"] not in all_B:
        all_B.append(r["B"])
    if r["C"] not in all_C:
        all_C.append(r["C"])

print("\nUnique A's in good triples:", all_A)
print("Unique B's in good triples:", all_B)
print("Unique C's in good triples:", all_C)


# If you also want to inspect frustrated ones:
# print("\n--- First 50 frustrated triples ---")
# for r in frust[:50]:
#     print(f'A: {r["A"]}, B: {r["B"]}, C: {r["C"]} | χ_AB={r["chi_AB"]:.6f}, χ_BC={r["chi_BC"]:.6f}, χ_AC={r["chi_AC"]:.6f}')


Non-frustrated triples: 538
Frustrated triples: 372
Triples within bounds: 117
A: 1a, B: 2a, C: 4a | χ_AB=0.204375, χ_BC=0.021306, χ_AC=0.111919 
A: 1a, B: 2c, C: 4a | χ_AB=0.053467, χ_BC=0.015068, χ_AC=0.111919 
A: 1a, B: 2d, C: 4a | χ_AB=0.211046, χ_BC=0.013530, χ_AC=0.111919 
A: 1a, B: 3a, C: 4a | χ_AB=0.039447, χ_BC=0.015441, χ_AC=0.111919 
A: 1b, B: 2a, C: 4a | χ_AB=0.305747, χ_BC=0.021306, χ_AC=0.204665 
A: 1b, B: 2c, C: 4a | χ_AB=0.131517, χ_BC=0.015068, χ_AC=0.204665 
A: 1b, B: 2d, C: 4a | χ_AB=0.426163, χ_BC=0.013530, χ_AC=0.204665 
A: 1b, B: 3a, C: 4a | χ_AB=0.092024, χ_BC=0.015441, χ_AC=0.204665 
A: 1c, B: 2a, C: 4a | χ_AB=0.272956, χ_BC=0.021306, χ_AC=0.174405 
A: 1c, B: 2c, C: 4a | χ_AB=0.102434, χ_BC=0.015068, χ_AC=0.174405 
A: 1c, B: 2d, C: 4a | χ_AB=0.336444, χ_BC=0.013530, χ_AC=0.174405 
A: 1c, B: 3a, C: 4a | χ_AB=0.075912, χ_BC=0.015441, χ_AC=0.174405 
A: 1d, B: 2a, C: 4a | χ_AB=0.217361, χ_BC=0.021306, χ_AC=0.124471 
A: 1d, B: 2c, C: 4a | χ_AB=0.062688, χ_BC=0.015068

/var/folders/yl/sv3dx7zs13s3phj74920ys6w0000gn/T/ipykernel_55831/1651700460.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  vals = df.applymap(lambda s: float(s.split(" ± ")[0]))
/var/folders/yl/sv3dx7zs13s3phj74920ys6w0000gn/T/ipykernel_55831/1651700460.py:11: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  errs = df.applymap(lambda s: float(s.split(" ± ")[1]))


In [41]:
import numpy as np
import pandas as pd
import itertools

# ----------------------------
# Load + parse
# ----------------------------
df = pd.read_csv("chi_table_with_errors.csv", index_col=0)
vals = df.applymap(lambda s: float(s.split(" ± ")[0]))
errs = df.applymap(lambda s: float(s.split(" ± ")[1]))

def chi(i, j):      return float(vals.loc[i, j])
def chi_err(i, j):  return float(errs.loc[i, j])

# ----------------------------
# Your existing filters
# ----------------------------
def is_frustrated(A, B, C):
    # (your current definition)
    chi_ab = chi(A, B)
    chi_bc = chi(B, C)
    chi_ac = chi(A, C)
    return (chi_bc > chi_ab) and (chi_bc > chi_ac)

def within_bounds(A, B, C):
    chi_ab = chi(A, B)
    chi_bc = chi(B, C)
    chi_ac = chi(A, C)
    if chi_ab <= 0 or chi_bc <= 0 or chi_ac <= 0:
        return False

    first  = (chi_ab / chi_ac >= 1/32) and (chi_ab / chi_ac <= 32)
    second = (chi_ab / chi_bc >= 1)    and (chi_ab / chi_bc <= 32)
    third  = (chi_ac / chi_bc >= 1)    and (chi_ac / chi_bc <= 32)
    return first and second and third

# ----------------------------
# Gyroid feasibility via N-interval intersection
# ----------------------------
def gyroid_N_interval(chi_ab, chi_ac, chi_bc):
    """
    Gyroid windows:
      chi_AB*N in [40, 80]
      chi_AC*N in [50, 80]
      chi_BC*N in [0,  40]
    Return (Nmin, Nmax) if feasible, else None.
    """
    if chi_ab <= 0 or chi_ac <= 0:
        return None

    # AB and AC always give finite positive intervals
    Nmin_ab, Nmax_ab = 40.0/chi_ab, 90.0/chi_ab
    Nmin_ac, Nmax_ac = 50.0/chi_ac, 90.0/chi_ac

    # BC: need chi_bc >= 0; if chi_bc == 0 then BC constraint is always satisfied for any N
    if chi_bc < 0:
        return None
    else:
        Nmin_bc, Nmax_bc = 4.0/chi_bc, 40.0/chi_bc

    Nmin = max(Nmin_ab, Nmin_ac, Nmin_bc, 0.0)
    Nmax = min(Nmax_ab, Nmax_ac, Nmax_bc)

    if Nmin <= Nmax and np.isfinite(Nmin) and (Nmax > 0):
        return (Nmin, Nmax)
    return None

def gyroid_score(chi_ab, chi_ac, chi_bc, Nmin, Nmax):
    """
    Rank candidates. Bigger feasible N-range is better (more robust).
    Also reward hitting "nice" target values at mid-N:
      AB ~ 60, AC ~ 65, BC ~ 20 (roughly centers of your windows).
    """
    Nmid = 0.5*(Nmin + Nmax)

    ab_mid = chi_ab * Nmid
    ac_mid = chi_ac * Nmid
    bc_mid = chi_bc * Nmid

    width = (Nmax - Nmin) / max(Nmid, 1e-12)  # relative width

    # penalty for being far from centers (tunable)
    pen = abs(ab_mid - 70.0)/30.0 + abs(ac_mid - 75.0)/25.0 + abs(bc_mid - 25.0)/15.0

    # higher is better
    return width - 0.25*pen

# ----------------------------
# Enumerate triples (C in 4*, A,B in {1*,2*,3*})
# ----------------------------
all_names = list(vals.index)
C_candidates  = [n for n in all_names if n.startswith("4")]
AB_candidates = [n for n in all_names if n[0] in {"1", "2", "3"}]

good = []
for C in C_candidates:
    for A, B in itertools.permutations(AB_candidates, 2):
        if A == B:
            continue
        if is_frustrated(A, B, C):
            continue
        if not within_bounds(A, B, C):
            continue

        chi_ab = chi(A, B)
        chi_ac = chi(A, C)
        chi_bc = chi(B, C)

        interval = gyroid_N_interval(chi_ab, chi_ac, chi_bc)
        if interval is None:
            continue

        Nmin, Nmax = interval
        score = gyroid_score(chi_ab, chi_ac, chi_bc, Nmin, Nmax)
        Nmid = 0.5*(Nmin + Nmax)

        good.append({
            "A": A, "B": B, "C": C,
            "chi_AB": chi_ab, "chi_AC": chi_ac, "chi_BC": chi_bc,
            "Nmin": Nmin, "Nmax": Nmax, "Nmid": Nmid,
            "AB*Nmid": chi_ab*Nmid, "AC*Nmid": chi_ac*Nmid, "BC*Nmid": chi_bc*Nmid,
            "score": score,
        })

print(f"Gyroid-feasible triples (after your filters): {len(good)}")

# ----------------------------
# Show top candidates overall
# ----------------------------
good_sorted = sorted(good, key=lambda r: r["score"], reverse=True)

print("\n=== TOP GYROID CANDIDATES (overall) ===")
for r in good_sorted:
    print(
        f'A:{r["A"]}  B:{r["B"]}  C:{r["C"]} | '
        f'chi_AB={r["chi_AB"]:.4f}, chi_AC={r["chi_AC"]:.4f}, chi_BC={r["chi_BC"]:.4f} | '
        f'N in [{r["Nmin"]:.1f}, {r["Nmax"]:.1f}] (mid {r["Nmid"]:.1f}) | '
        f'AB*Nmid={r["AB*Nmid"]:.1f}, AC*Nmid={r["AC*Nmid"]:.1f}, BC*Nmid={r["BC*Nmid"]:.1f} | '
        f'score={r["score"]:.3f}'
    )

# ----------------------------
# (Optional) top candidates per C-block
# ----------------------------
print("\n=== TOP 10 PER C (4*) ===")
for C in C_candidates:
    rows = [r for r in good_sorted if r["C"] == C]
    if not rows:
        continue
    print(f"\n-- C = {C} (n={len(rows)}) --")
    for r in rows[:10]:
        print(
            f'A:{r["A"]}  B:{r["B"]} | '
            f'N[{r["Nmin"]:.1f},{r["Nmax"]:.1f}] mid {r["Nmid"]:.1f} | '
            f'ABN={r["AB*Nmid"]:.1f}, ACN={r["AC*Nmid"]:.1f}, BCN={r["BC*Nmid"]:.1f} | '
            f'score={r["score"]:.3f}'
        )


Gyroid-feasible triples (after your filters): 24

=== TOP GYROID CANDIDATES (overall) ===
A:1b  B:2c  C:4a | chi_AB=0.1315, chi_AC=0.2047, chi_BC=0.0151 | N in [304.1, 439.7] (mid 371.9) | AB*Nmid=48.9, AC*Nmid=76.1, BC*Nmid=5.6 | score=-0.146
A:1g  B:2c  C:4a | chi_AB=0.1134, chi_AC=0.1896, chi_BC=0.0151 | N in [352.6, 474.7] (mid 413.7) | AB*Nmid=46.9, AC*Nmid=78.4, BC*Nmid=6.2 | score=-0.244
A:1c  B:2c  C:4a | chi_AB=0.1024, chi_AC=0.1744, chi_BC=0.0151 | N in [390.5, 516.0] (mid 453.3) | AB*Nmid=46.4, AC*Nmid=79.1, BC*Nmid=6.8 | score=-0.263
A:1f  B:2a  C:4c | chi_AB=0.0179, chi_AC=0.0125, chi_BC=0.0015 | N in [3992.0, 5019.2] (mid 4505.6) | AB*Nmid=80.8, AC*Nmid=56.4, BC*Nmid=6.6 | score=-0.355
A:1e  B:1f  C:4c | chi_AB=0.1032, chi_AC=0.1851, chi_BC=0.0125 | N in [387.7, 486.3] (mid 437.0) | AB*Nmid=45.1, AC*Nmid=80.9, BC*Nmid=5.5 | score=-0.366
A:1a  B:1f  C:4c | chi_AB=0.1179, chi_AC=0.2122, chi_BC=0.0125 | N in [339.4, 424.2] (mid 381.8) | AB*Nmid=45.0, AC*Nmid=81.0, BC*Nmid=4.

/var/folders/yl/sv3dx7zs13s3phj74920ys6w0000gn/T/ipykernel_55831/1300690298.py:9: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  vals = df.applymap(lambda s: float(s.split(" ± ")[0]))
/var/folders/yl/sv3dx7zs13s3phj74920ys6w0000gn/T/ipykernel_55831/1300690298.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  errs = df.applymap(lambda s: float(s.split(" ± ")[1]))


In [36]:
0.35/0.12 * 7

20.416666666666664